In [0]:
%sql
CREATE TABLE IF NOT EXISTS identifier(:catalog || '.gold.product_analytics') (
    product_id INT,
    product_name STRING,
    category STRING,
    supplier_id INT,
    supplier_name STRING,
    total_units_sold BIGINT,
    total_revenue DECIMAL(14,2),
    category_revenue_share_pct DECIMAL(8,2),
    revenue_percentile DECIMAL(8,4)
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW product_sales_agg AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.supplier_id,
    sup.supplier_name,
    SUM(s.quantity)     AS total_units_sold,
    SUM(s.sale_amount)  AS total_revenue,
    ROUND(
        SUM(s.sale_amount) / SUM(SUM(s.sale_amount)) OVER (PARTITION BY p.category) * 100, 2
    ) AS category_revenue_share_pct,
    ROUND(
        PERCENT_RANK() OVER (PARTITION BY p.category ORDER BY SUM(s.sale_amount)), 4
    ) AS revenue_percentile
FROM identifier(:catalog || '.silver.sales_clean') s
JOIN identifier(:catalog || '.silver.products_scd2')  p   ON s.product_id = p.product_id AND p.is_current = true
JOIN identifier(:catalog || '.silver.suppliers_scd2')  sup ON p.supplier_id = sup.supplier_id AND sup.is_current = true
GROUP BY p.product_id, p.product_name, p.category, p.supplier_id, sup.supplier_name;

In [0]:
%sql
INSERT OVERWRITE identifier(:catalog || '.gold.product_analytics')
SELECT product_id, product_name, category, supplier_id, supplier_name,
       total_units_sold, total_revenue, category_revenue_share_pct, revenue_percentile
FROM product_sales_agg;

In [0]:
%sql
SELECT product_id, product_name, category, total_revenue, category_rank
FROM (
    SELECT *, RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS category_rank
    FROM identifier(:catalog || '.gold.product_analytics')
)
WHERE category_rank <= 5
ORDER BY category, category_rank;

In [0]:
%sql
SELECT
    product_id, product_name, price, version, effective_date, end_date, is_current,
    LAG(price) OVER (PARTITION BY product_id ORDER BY version) AS previous_price,
    price - LAG(price) OVER (PARTITION BY product_id ORDER BY version) AS price_change
FROM identifier(:catalog || '.silver.products_scd2')
ORDER BY product_id, version;

In [0]:
%sql
SELECT
    supplier_id, supplier_name, product_id, product_name, total_revenue,
    SUM(total_revenue) OVER (PARTITION BY supplier_id) AS supplier_total_revenue,
    ROUND(AVG(total_revenue) OVER (PARTITION BY supplier_id), 2) AS supplier_avg_product_revenue,
    COUNT(product_id) OVER (PARTITION BY supplier_id) AS supplier_product_count
FROM identifier(:catalog || '.gold.product_analytics')
ORDER BY supplier_total_revenue DESC, supplier_id;